# Parler-TTS: Control Voice with Natural Language

## What is Parler-TTS?
Parler-TTS lets you **describe the voice you want in plain English**, and it generates speech matching that description.

Instead of picking from a fixed list of voices, you write things like:
> *"A young woman speaks quickly with excitement. Clear studio recording."*

### How it works (simplified)
```
┌──────────────────┐    ┌──────────────┐    ┌──────────────┐
│ Voice Description│───▶│   Encoder    │───▶│              │
│ "calm male..."   │    │ (text→embed) │    │   Decoder    │───▶ 🔊 Audio
│                  │    └──────────────┘    │ (DAC codec)  │
│ Text to speak    │───▶│   Encoder    │───▶│              │
│ "Hello world"    │    │ (text→embed) │    └──────────────┘
└──────────────────┘    └──────────────┘
```

**Architecture**: Encoder-decoder transformer + DAC audio codec  
**Training data**: 45K hours of narrated audiobooks  
**Key paper**: [Parler-TTS (2024)](https://huggingface.co/papers/2402.01912)

---
**Kaggle Setup**: GPU T4 x2 → Accelerator dropdown

## Step 1: Install & Load

In [ ]:
!pip install -q parler-tts transformers accelerate soundfile

In [ ]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
import soundfile as sf
from IPython.display import Audio, display
import time

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Load the mini model (~1GB) - good balance of speed and quality
# Other options: parler-tts/parler-tts-large-v1 (better quality, slower)

MODEL_NAME = "parler-tts/parler-tts-mini-v1"

print(f"Loading {MODEL_NAME}...")
t0 = time.time()

model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
print(f"Sample rate: {model.config.sampling_rate} Hz")

## Step 2: Your First Speech Generation

Two inputs are needed:
1. **`description`** — how the voice should sound (gender, speed, pitch, quality)
2. **`prompt`** — the actual text to speak

In [ ]:
description = (
    "A female speaker delivers a slightly expressive and animated speech "
    "with a moderate speed and pitch. The recording is of very high quality "
    "with clear audio and no background noise."
)

prompt = "Hey, welcome to this text-to-speech demo! Let me show you how natural AI voices can sound."

# Tokenize both inputs
input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)
prompt_input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

print(f"Description tokens: {input_ids.shape[1]}")
print(f"Prompt tokens: {prompt_input_ids.shape[1]}")

# Generate!
t0 = time.time()
with torch.no_grad():
    generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
gen_time = time.time() - t0

audio_arr = generation.cpu().numpy().squeeze()
audio_duration = len(audio_arr) / model.config.sampling_rate

print(f"\nGeneration time: {gen_time:.1f}s")
print(f"Audio duration: {audio_duration:.1f}s")
print(f"Real-time factor: {gen_time/audio_duration:.2f}x (< 1.0 = faster than real-time)")

display(Audio(audio_arr, rate=model.config.sampling_rate))

## Step 3: Experiment with Voice Descriptions

The magic of Parler-TTS is in the description. Here are the key attributes you can control:

| Attribute | Examples |
|-----------|----------|
| **Gender** | male, female |
| **Age** | young, middle-aged, old |
| **Speed** | very slowly, slowly, moderate speed, fast |
| **Pitch** | low pitch, moderate pitch, high pitch |
| **Emotion** | calm, animated, expressive, monotone |
| **Quality** | very high quality, clear audio, studio recording |

In [ ]:
text = "The quick brown fox jumps over the lazy dog. This sentence contains every letter of the alphabet."

experiments = {
    "Deep Male Voice": (
        "A male speaker with a deep, calm voice speaks slowly and clearly. "
        "Very high quality studio recording with no background noise."
    ),
    "Energetic Female": (
        "A young woman speaks quickly with enthusiasm and energy. "
        "High pitch, very expressive. Clear recording."
    ),
    "Storyteller": (
        "An older gentleman narrates in a warm, storytelling tone at a measured pace. "
        "Moderate pitch. Very high quality recording."
    ),
    "Monotone (bad example)": (
        "A speaker delivers speech in a very monotone and flat way. "
        "Slow speed, low pitch. Slightly noisy recording."
    ),
}

for name, desc in experiments.items():
    input_ids = tokenizer(desc, return_tensors="pt").input_ids.to(device)
    prompt_input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)
    
    with torch.no_grad():
        generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
    audio_arr = generation.cpu().numpy().squeeze()
    
    print(f"\n{'='*60}")
    print(f"🎙️ {name}")
    print(f"Description: {desc[:80]}...")
    display(Audio(audio_arr, rate=model.config.sampling_rate))

## Step 4: Long Text Generation

For longer texts, split into sentences and concatenate. The model works best with short segments (1-2 sentences).

In [ ]:
import numpy as np
import re

def generate_long_text(model, tokenizer, description, text, device="cuda:0"):
    """Split long text into sentences and generate audio for each."""
    # Split on sentence boundaries
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if s.strip()]
    
    audio_chunks = []
    desc_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)
    
    for i, sent in enumerate(sentences):
        print(f"  Generating [{i+1}/{len(sentences)}]: {sent[:50]}...")
        prompt_ids = tokenizer(sent, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            gen = model.generate(input_ids=desc_ids, prompt_input_ids=prompt_ids)
        audio_chunks.append(gen.cpu().numpy().squeeze())
        # Add small silence between sentences
        silence = np.zeros(int(model.config.sampling_rate * 0.3))  # 300ms pause
        audio_chunks.append(silence)
    
    return np.concatenate(audio_chunks)

long_text = (
    "Artificial intelligence has made remarkable progress in speech synthesis. "
    "Modern models can produce natural-sounding voices that are nearly indistinguishable from real humans. "
    "This has applications in accessibility, education, and entertainment."
)

description = "A female speaker with a clear, professional voice at moderate speed. Very high quality recording."

print("Generating long text...")
full_audio = generate_long_text(model, tokenizer, description, long_text, device)
print(f"\nTotal duration: {len(full_audio)/model.config.sampling_rate:.1f}s")
display(Audio(full_audio, rate=model.config.sampling_rate))

# Save
sf.write("parler_long_output.wav", full_audio, model.config.sampling_rate)
print("Saved to parler_long_output.wav")

## Key Takeaways

**Pros:**
- No reference audio needed — just describe the voice
- Fast inference on GPU
- Good for prototyping different voice styles

**Cons:**
- English only (as of v1)
- Can't clone a specific person's voice
- Voice consistency varies between generations

**When to use**: Quick experiments, English narration, accessibility tools

**Next**: Try `02_bark_tts/` for multilingual + expressive speech, or `03_xtts_v2/` for voice cloning.